# 0316: Parse Acc Verifier

## Prelude

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
from os import environ
import apfel as apfel
import statsmodels.api as sm

In [3]:
dataset = pl.read_parquet(f"{environ['DATA_PATH_BASE']}/v2/babylm/0310/parsed-roberta.pos-lemma.parquet")

In [4]:
dataset

index,text,split,parse,lemma,pos
u32,str,enum,struct[3],list[str],list[str]
0,"""aboo .""","""train""",null,null,null
1,"""kee .""","""train""",null,null,null
2,"""hey .""","""train""",null,null,null
3,"""smile ?""","""train""",null,null,null
4,"""hm hm .""","""train""","{[""root"", ""dep"", ""punct""],[0, 1, 1],[""hm"", ""hm"", "".""]}","[""hm"", ""hm"", "".""]","[""INTJ"", ""INTJ"", ""PUNCT""]"
…,…,…,…,…,…
209995,"""oh my heavens ?""","""test""","{[""root"", ""poss"", … ""punct""],[0, 3, … 1],[""oh"", ""my"", … ""?""]}","[""oh"", ""my"", … ""?""]","[""INTJ"", ""PRON"", … ""PUNCT""]"
209996,"""oh hi jessica .""","""test""","{[""root"", ""dep"", … ""punct""],[0, 1, … 3],[""oh"", ""hi"", … "".""]}","[""oh"", ""hi"", … "".""]","[""INTJ"", ""INTJ"", … ""PUNCT""]"
209997,"""are you helping the babysitter…","""test""","{[""aux"", ""nsubj"", … ""punct""],[3, 3, … 3],[""are"", ""you"", … ""?""]}","[""be"", ""you"", … ""?""]","[""AUX"", ""PRON"", … ""PUNCT""]"


## Results

In [5]:
dataset_result = pl.read_ndjson("/Users/tomyxw/Documents/Projects/work/verb-overgen/repo/src/verb_overgen/prep/asset/parse_verification/20250317T151039.jsonl")

In [10]:
dataset_result.select(pl.col("is_good_english").value_counts().struct.unnest()).select(
    pl.col("is_good_english").replace({
        "y": "good",
        "p": "bad_punct",
        "g": "ungrammatical",
        "t": "bad_tokenization",
        "m": "paralinguistic",
    }),
    pl.col("count")
)


is_good_english,count
str,u32
"""good""",50
"""ungrammatical""",9
"""paralinguistic""",14
"""bad_punct""",6
"""bad_tokenization""",25


In [13]:
dataset_result.filter(pl.col("is_good_english") == "y").select(
    pl.col("is_parse_ok").value_counts().struct.unnest()
).with_columns(
    pl.col(name="is_parse_ok").replace({
        "y": "correct",
        "p": "partially_incorrect",
        "n": "incorrect",
    }
))


is_parse_ok,count
str,u32
"""incorrect""",6
"""correct""",43
"""partially_incorrect""",1
